# Consumo del modelo y app Gradio — PhysioVision Ex1

**PhysioVision · Diplomado Modulo 5 · notebook 2 de 2: consumo**

El primer notebook ([`modelo_xgboost_ex1.ipynb`](modelo_xgboost_ex1.ipynb)) entrena y
resguarda. Este **no entrena nada**: carga los artefactos de `models/` y los usa, hasta
levantar la aplicacion.

| # | Seccion | Que hace |
|---|---|---|
| 1 | Configuracion | rutas y comprobacion de que los artefactos existen |
| 2 | Carga de los artefactos | `ExerciseClassifier` + contrato de variables |
| 3 | Datos de referencia | `df_reps` desde `data/datasets/` (para las demos) |
| 4 | Inferencia extremo a extremo | video nuevo -> repeticiones -> clase |
| 5 | Redaccion con Gemini | la consigna, dicha como una persona |
| 6 | Voz con Text-to-Speech | la consigna, en audio |
| 7 | **PhysioVision en Gradio** | la app, en su forma mas simple |

**Requisito**: `models/xgboost_model.json` y `models/feature_contract.json`. Los genera
la seccion 13 del primer notebook. Si no estan, la celda 1 lo dice y para.

Lo que hace cada pieza, y lo que no:

| Pieza | Decide |
|---|---|
| XGBoost | la clase de la repeticion |
| `knowledge_base/ejercicios.json` | que recomendar para esa clase |
| Gemini | como decirlo |
| Cloud TTS | como suena |

> **Descargo clinico.** Material academico. No constituye diagnostico ni sustituye el
> criterio de un profesional de la salud.

---
## 0. Entorno y reproducibilidad

Igual que el notebook de entrenamiento, esta libreta corre en local y en **Google
Colab**. La celda 0.1 trae el código, instala lo que falte y comprueba qué hay
disponible.

Lo esencial **viaja con el repositorio**: el modelo, su contrato de variables y
los umbrales están versionados, así que las secciones 2, 5, 6 y 7 —carga,
redacción, voz e interfaz— funcionan en Colab tal cual.

Lo que no viaja son los datos de los pacientes:

| Sección | Necesita | Sin ello |
|---|---|---|
| 3 y los ejemplos de 5 | dataset de repeticiones | se saltan con aviso |
| 4.1 | un vídeo original | se salta con aviso |
| 2, 6, 7 | solo `models/` y claves opcionales | funcionan siempre |

**Convención de este notebook**: las celdas que dependen de datos empiezan con
`if not HAY_...`, avisan y siguen. Ninguna interrumpe la ejecución.

En Colab la interfaz de la sección 7 se abre con un enlace público temporal
(`share=True`), porque no hay navegador local al que asomarse.

In [ ]:
# 0.1 Entorno: codigo, dependencias y datos disponibles.
#     Idempotente y sin magias de Jupyter, para que valga igual en Colab, en
#     Jupyter local y ejecutado como script.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/daff-250900/phyisiovision.git"

#: Lo que Colab no trae. Versiones del entorno donde se valido el modelo
#: (requirements.lock): mediapipe fija los landmarks y xgboost el clasificador.
PAQUETES = {"mediapipe": "mediapipe==0.10.35", "xgboost": "xgboost==3.2.0",
            "gradio": "gradio==6.20.0", "google.genai": "google-genai==2.14.0"}

EN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

if EN_COLAB:
    destino = Path("/content/phyisiovision")
    if not destino.exists():
        print("Clonando el repositorio...")
        subprocess.run(["git", "clone", "--depth", "1", "--quiet", REPO, str(destino)],
                       check=True)
    os.chdir(destino / "entrenamiento")

    faltan = [pin for modulo, pin in PAQUETES.items()
              if importlib.util.find_spec(modulo.split(".")[0]) is None]
    if faltan:
        print(f"Instalando {', '.join(faltan)} (un par de minutos)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *faltan],
                       check=True)

NB_DIR = Path.cwd()
BASE_DIR = NB_DIR.parent if NB_DIR.name == "entrenamiento" else NB_DIR
if not (BASE_DIR / "src").exists() and (BASE_DIR.parent / "src").exists():
    BASE_DIR = BASE_DIR.parent
sys.path.insert(0, str(BASE_DIR))

# Los datos pueden estar en el repositorio o montados aparte (Drive, un disco).
DATOS_DIR = Path(os.environ.get("PHYSIOVISION_DATOS_EXTERNOS", BASE_DIR / "data"))
VIDEO_DIR = DATOS_DIR / "videos" / "Ex1"
DATASET_DIR = DATOS_DIR / "datasets"
MODEL_DIR = BASE_DIR / "models"

HAY_VIDEOS = VIDEO_DIR.exists() and any(VIDEO_DIR.glob("*.mp4"))
HAY_DATASET = (DATASET_DIR / "ex1_repeticiones.csv").exists()
HAY_MODELO = (MODEL_DIR / "xgboost_model.json").exists()

print(f"Entorno  : {'Google Colab' if EN_COLAB else 'local'}")
print(f"Proyecto : {BASE_DIR}")
print()
for etiqueta, hay, secciones in [
    ("modelo y contrato", HAY_MODELO, "2, 4, 5, 6 y 7 (lo esencial)"),
    ("dataset de repeticiones", HAY_DATASET, "3 y los ejemplos de 5"),
    ("videos originales", HAY_VIDEOS, "4.1 (inferencia sobre un video)"),
]:
    print(f"  {'si' if hay else 'NO':3s}  {etiqueta:26s} {secciones}")

if not HAY_MODELO:
    print()
    print("Sin modelo entrenado: ejecuta antes modelo_xgboost_ex1.ipynb, o clona")
    print("el repositorio completo, donde models/ va versionado.")

---
## 1. Configuracion

Solo rutas: ni semillas de entrenamiento ni parametros del pipeline, porque aqui no se
reentrena nada. Los umbrales de segmentacion y de las reglas se leen de
`models/umbrales_ex1.json`, que exporto el primer notebook, para que la inferencia use
exactamente los mismos que el entrenamiento.

La celda **para con un error si faltan los artefactos**: es preferible eso a que
`ExerciseClassifier` caiga en silencio a las reglas y el resto del notebook parezca
funcionar sin usar el modelo.

In [1]:
from __future__ import annotations

import json
import os
import sys
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ---- Ejercicio y artefactos -------------------------------------------------
# Las rutas y la deteccion de entorno vienen de la seccion 0.1.
# Dos identificadores distintos del mismo ejercicio, y conviene no confundirlos:
EJERCICIO = "Ex1"                          # carpeta de datos y sufijo de artefactos
EJERCICIO_KB = "elevacion_lateral_hombro"  # clave en knowledge_base/ejercicios.json

RUTA_MODELO = MODEL_DIR / "xgboost_model.json"
RUTA_CONTRATO = MODEL_DIR / "feature_contract.json"

FPS_NOMINAL = 30.0
# El entrenamiento uso la variante "heavy" (proceso por lotes, primaba la precision).
# Aqui hay una persona esperando delante de la pantalla, asi que "full".
VARIANTE_MP = os.environ.get("PV_VARIANTE_MP", "full")

print(f"Ejercicio : {EJERCICIO}")
print(f"MediaPipe : variante {VARIANTE_MP}")

faltan = [p.name for p in (RUTA_MODELO, RUTA_CONTRATO) if not p.exists()]
if faltan:
    # No se lanza excepcion: el notebook tiene que poder leerse y ejecutarse
    # entero aunque falte el modelo. Las celdas que lo necesitan se saltan.
    print(f"Faltan artefactos en models/: {faltan}")
    print("Ejecuta antes modelo_xgboost_ex1.ipynb (seccion 13).")

print("Artefactos:")
for p in (RUTA_MODELO, RUTA_CONTRATO, MODEL_DIR / "umbrales_ex1.json"):
    estado = f"{p.stat().st_size / 1024:.0f} KB" if p.exists() else "AUSENTE (opcional)"
    print(f"  {p.name:24s} {estado}")

Proyecto : /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio
Videos   : /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio/data/videos/Ex1  (26 archivos)
MediaPipe: variante full

Artefactos encontrados:
  xgboost_model.json       483 KB
  feature_contract.json    3 KB
  umbrales_ex1.json        0 KB


---
## 2. Carga de los artefactos entrenados

`ExerciseClassifier` (`src/classifier.py`) ya hace por su cuenta lo unico delicado de
esta parte: **lee el contrato de variables desde `models/feature_contract.json`** en vez
de tener la lista escrita a mano. Eso garantiza que el orden y los nombres de las
columnas sean exactamente los que vio el modelo al entrenar. Si no coinciden, el
constructor lanza `ValueError` en lugar de predecir en silencio sobre columnas
desalineadas, que es el fallo mas caro posible aqui.

Si el modelo no estuviera, la clase no falla: cae a las reglas biomecanicas y lo
declara en `source: "reglas"`. Por eso la celda comprueba `usa_modelo`.

In [2]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    from src.classifier import ExerciseClassifier
    from src.config import settings
    from src.rag import KnowledgeBase

    clf = ExerciseClassifier()          # lee models/feature_contract.json por su cuenta
    kb = KnowledgeBase()

    contrato = json.loads(RUTA_CONTRATO.read_text(encoding="utf-8"))
    FEATURES = clf.FEATURE_NAMES
    CLASES = clf.LABELS
    CLASE_A_ID = {v: k for k, v in CLASES.items()}
    MEDIANAS = clf.medianas

    if not clf.usa_modelo:
        raise RuntimeError(
            "ExerciseClassifier cayo a las reglas: no logro cargar el modelo. "
            f"Revisa {RUTA_MODELO}."
        )

    print(f"modelo      : {settings.model_path}")
    print(f"contrato    : v{contrato['version']}, creado {contrato['creado'][:10]}")
    print(f"ejercicio   : {contrato['ejercicio']}")
    print(f"variables   : {len(FEATURES)}")
    print(f"clases      : {CLASES}")
    print(f"macro-F1    : {contrato['metricas_loso']['macro_f1']:.3f} "
          f"IC95% {contrato['ic95_macro_f1']}")
    print(f"etiquetado  : {contrato['entrenamiento']['etiquetado']}")
    print(f"umbrales    : ROM_MINIMO {clf.rom_minimo:.0f}  TRONCO_LIMITE {clf.tronco_limite:.0f}")
    print(f"\nknowledge_base: {kb.path.name}, ejercicio '{EJERCICIO_KB}'")
    print(f"  clases con recomendacion: {list(kb.data[EJERCICIO_KB]['errores'])}")

modelo      : /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio/models/xgboost_model.json
contrato    : v1.0.0, creado 2026-08-01
ejercicio   : Ex1
variables   : 26
clases      : {0: 'rango_insuficiente', 1: 'correcto', 2: 'compensacion_tronco'}
macro-F1    : 0.869 IC95% [0.7912, 0.9234]
etiquetado  : manual_revisado
umbrales    : ROM_MINIMO 77  TRONCO_LIMITE 14

knowledge_base: ejercicios.json, ejercicio 'elevacion_lateral_hombro'
  clases con recomendacion: ['rango_insuficiente', 'compensacion_tronco', 'correcto']


---
## 3. Datos de referencia desde disco

No hace falta reextraer landmarks ni reentrenar: el primer notebook dejo las
repeticiones y sus etiquetas en `data/datasets/`. Se reconstruye `df_reps` para tener
repeticiones reales con las que probar la redaccion (seccion 5) sin depender de
procesar un video.

La etiqueta se guarda **por gesto** —un gesto es la misma repeticion vista por las dos
camaras—, asi que hay que reunirla con las filas por repeticion.

In [3]:
# Requiere el dataset de repeticiones (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_DATASET:
    print("Se salta: falta el dataset de repeticiones.")
else:
    def cargar_repeticiones() -> pd.DataFrame:
        reps = pd.read_csv(DATASET_DIR / "ex1_repeticiones.csv")
        etiquetas = pd.read_csv(DATASET_DIR / "ex1_etiquetas.csv")
        df = reps.merge(etiquetas[["gesto_id", "etiqueta"]], on="gesto_id", how="left")
        sin_etiqueta = int(df.etiqueta.isna().sum())
        if sin_etiqueta:
            print(f"AVISO: {sin_etiqueta} repeticiones sin etiqueta, se descartan.")
            df = df[df.etiqueta.notna()]
        return df.reset_index(drop=True)


    df_reps = cargar_repeticiones()
    df_reps["y"] = df_reps.etiqueta.map(CLASE_A_ID)
    print(f"{len(df_reps)} repeticiones, {df_reps.sujeto.nunique()} sujetos, "
          f"{df_reps.gesto_id.nunique()} gestos")
    print(df_reps.etiqueta.value_counts().to_string())

    # Prueba de humo del contrato: una repeticion real pasada por el clasificador.
    # Si `source` no es "xgboost", el modelo no se cargo y todo lo que sigue seria
    # el comportamiento degradado, no el del modelo entrenado.
    ejemplo = clf.predict(df_reps.iloc[0].to_dict())
    print(f"\nPrimera repeticion ({df_reps.iloc[0].sujeto}, etiqueta "
          f"{df_reps.iloc[0].etiqueta}):")
    for k, v in ejemplo.items():
        print(f"  {k}: {v}")
    assert ejemplo["source"] == "xgboost"

334 repeticiones, 13 sujetos, 167 gestos
etiqueta
correcto               186
compensacion_tronco    134
rango_insuficiente      14

Primera repeticion (PM_000, etiqueta compensacion_tronco):
  class_id: 2
  label: compensacion_tronco
  confidence: 0.91883784532547
  probabilities: {'rango_insuficiente': 0.004112705588340759, 'correcto': 0.07704946398735046, 'compensacion_tronco': 0.91883784532547}
  source: xgboost


---
## 4. Inferencia extremo a extremo: de un video a una clase

**La unidad de prediccion es la repeticion, no el video.** El modelo nunca vio un video
completo, asi que hay que reproducir el mismo camino del entrenamiento —landmarks,
angulos, suavizado, segmentacion, variables por repeticion— y solo entonces predecir,
una vez por repeticion.

Las funciones geometricas son las mismas que uso el entrenamiento (`src/features_ex1.py`),
no copias: si divergieran, el modelo recibiria variables calculadas de otra forma que
las que aprendio.

**Agregacion `peor_caso`.** Para el resumen de la serie no se promedia: si al menos un
tercio de las repeticiones falla, se reporta el error mas frecuente. Un error en varias
repeticiones es informacion clinica relevante y promediar lo borraria.

Dos supuestos del video subido, que conviene tener presentes:

- se asume **vista frontal** (`es_lateral = 0`), como la Camera17 del entrenamiento;
- el **lado activo** se detecta en el propio video, por mayor recorrido angular.

In [4]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    from src.features_ex1 import (
        detectar_lado, features_repeticion, segmentar, series_angulares, suavizar)
    from src.pose_detector import extraer_landmarks


    def repeticiones_del_video(video_path: str | Path,
                               variante: str = VARIANTE_MP) -> tuple[pd.DataFrame, str, float]:
        """video -> una fila por repeticion, con las variables del contrato."""
        df_lm = extraer_landmarks(video_path, variante=variante)

        captura = cv2.VideoCapture(str(video_path))
        fps = captura.get(cv2.CAP_PROP_FPS) or FPS_NOMINAL
        captura.release()

        lado, _, _ = detectar_lado(df_lm)
        s = series_angulares(df_lm, lado)
        s["abduccion_suave"] = suavizar(s["abduccion_hombro"].where(s.valido), fps)
        reps = segmentar(s["abduccion_suave"].to_numpy(), fps)
        if not reps:
            raise ValueError("No se detectaron repeticiones completas en el video.")

        filas = [features_repeticion(s, r, fps, i, len(reps)) for i, r in enumerate(reps)]
        filas = [f for f in filas if f]
        if not filas:
            raise ValueError("Las repeticiones detectadas no superaron los filtros.")

        X = pd.DataFrame(filas)
        X["es_lateral"] = 0   # el video de la app se asume vista frontal
        return X, lado, fps


    def predecir_video(video_path: str | Path, agregacion: str = "peor_caso",
                       variante: str = VARIANTE_MP) -> dict:
        """Prediccion por repeticion + una clase agregada para la serie."""
        X, lado, fps = repeticiones_del_video(video_path, variante)

        # clf.predict ordena las columnas segun el contrato e imputa lo que falte con
        # las medianas del entrenamiento: no hay que alinear nada a mano.
        por_rep = [clf.predict(fila) for fila in X.to_dict("records")]
        pred = np.array([r["class_id"] for r in por_rep])

        id_correcto = CLASE_A_ID["correcto"]
        errores = pred[pred != id_correcto]
        if agregacion == "peor_caso" and len(errores) >= max(1, len(pred) / 3):
            clase = int(pd.Series(errores).mode()[0])
        elif agregacion == "peor_caso":
            clase = id_correcto
        else:
            clase = int(pd.Series(pred).mode()[0])

        detalle = [
            {"idx": i,
             "label": r["label"],
             "confidence": r["confidence"],
             "rom_max": float(X.rom_max.iloc[i]),
             "tronco_max": float(X.tronco_max.iloc[i]),
             "duracion_s": float(X.duracion_s.iloc[i])}
            for i, r in enumerate(por_rep)
        ]

        return {
            "class_id": clase,
            "label": CLASES[clase],
            "confidence": float(np.mean([r["probabilities"][CLASES[clase]] for r in por_rep])),
            "probabilities": {c: float(np.mean([r["probabilities"][c] for r in por_rep]))
                              for c in CLASES.values()},
            "source": por_rep[0]["source"],
            "n_repeticiones": len(por_rep),
            "correctas": int((pred == id_correcto).sum()),
            "por_repeticion": [r["label"] for r in por_rep],
            "detalle": detalle,
            "lado": lado,
            "fps": float(fps),
            "rom_max": float(X.rom_max.max()),
            "rom_medio": float(X.rom_max.mean()),
        }

### 4.1 Prueba sobre un video del dataset

In [5]:
# Requiere los videos originales (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_VIDEOS:
    print("Se salta: falta los videos originales.")
else:
    # Prueba sobre un video real del dataset. Tarda lo que tarde MediaPipe en recorrer
    # el video entero (decenas de segundos), asi que se hace con uno solo.
    videos = sorted(VIDEO_DIR.glob("*.mp4"))
    if videos:
        video_prueba = videos[0]
        print(f"Prediciendo sobre {video_prueba.name} ...")
        salida = predecir_video(video_prueba)
        for k, v in salida.items():
            if k != "detalle":
                print(f"  {k}: {v}")
        print("\n  por repeticion:")
        print(pd.DataFrame(salida["detalle"]).to_string(index=False))
    else:
        print(f"No hay videos en {VIDEO_DIR}: se salta la prueba. "
              "La seccion 7 acepta cualquier video subido.")

Prediciendo sobre PM_000-Camera17-30fps.mp4 ...


I0000 00:00:1785597615.204837 2382509 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 13, prefix = pthread-default
I0000 00:00:1785597615.269417 2382509 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1785597615.311322 2382512 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785597615.322173 2382520 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785597615.370247 2382520 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  class_id: 2
  label: compensacion_tronco
  confidence: 0.953927606344223
  probabilities: {'rango_insuficiente': 0.006283867638558149, 'correcto': 0.03978853113949299, 'compensacion_tronco': 0.953927606344223}
  source: xgboost
  n_repeticiones: 4
  correctas: 0
  por_repeticion: ['compensacion_tronco', 'compensacion_tronco', 'compensacion_tronco', 'compensacion_tronco']
  lado: right
  fps: 30.0
  rom_max: 168.2269980646933
  rom_medio: 159.90205435834693

  por repeticion:
 idx               label  confidence    rom_max  tronco_max  duracion_s
   0 compensacion_tronco    0.975591 141.009964   17.914187   10.066667
   1 compensacion_tronco    0.949656 168.226998   17.867480    7.800000
   2 compensacion_tronco    0.928685 164.036693   17.716183    7.966667
   3 compensacion_tronco    0.961778 166.334562   17.104762    7.533333


---
## 5. Redaccion de la realimentacion con Google Gemini

El modelo dice *que* falla; `knowledge_base/ejercicios.json` dice *que hacer*.
Ninguno de los dos suena a persona: el texto del JSON es identico para todas las
repeticiones de la misma clase, diga 95 grados de rango o 140.

Gemini se ocupa solo de eso: **reformular** la recomendacion ya validada usando
las metricas concretas de esa repeticion. No clasifica, no diagnostica y no
inventa consejo clinico. La division de trabajo es:

| Pieza | Decide |
|---|---|
| XGBoost | la clase de la repeticion |
| `ejercicios.json` | que recomendar para esa clase |
| Gemini | como decirlo |

> **Por que importa la restriccion.** Un modelo de lenguaje generando consejo de
> rehabilitacion por su cuenta produce texto plausible y sin respaldo, y aqui lo
> lee alguien moviendose con un hombro lesionado. Por eso la instruccion de
> sistema se lo prohibe explicitamente y la advertencia de seguridad se copia
> literal del JSON, sin pasar por el modelo.

Esta seccion sirve para **afinar los prompts** contra repeticiones reales del
dataset antes de que lleguen a un paciente. Los prompts viven en
`src/gemini_feedback.py`, el mismo modulo que usa la app: lo que ajustes aqui es
lo que se ejecutara en produccion.

In [6]:
import importlib
import os

from src import config as _config
from src import gemini_feedback as _gemini

# Importar src.config carga .env. Pero si este kernel ya importo el modulo antes
# de que .env existiera, Python lo tiene cacheado en sys.modules y volver a
# ejecutar esta celda seguiria usando la version vieja, sin la clave. Recargar
# resuelve el caso sin obligar a reiniciar el kernel.
if not (os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")):
    importlib.reload(_config)
    importlib.reload(_gemini)

from src.gemini_feedback import (
    INSTRUCCION_SISTEMA, PROMPT_REPETICION, PROMPT_RESUMEN, RedactorGemini)

redactor = RedactorGemini()

ruta_env = _config.BASE_DIR / ".env"
print(f".env: {'encontrado' if ruta_env.exists() else 'NO existe'} en {ruta_env}")
print(f"claves cargadas: {list(_config.DOTENV_CARGADO) or 'ninguna'}")

if redactor.disponible:
    print(f"\nGemini disponible — modelo {redactor.modelo}")
else:
    print(f"\nGemini NO disponible: {redactor.motivo_no_disponible}")
    print("\nQue revisar, por orden:")
    print("  1. Que exista .env en la raiz con GEMINI_API_KEY=<tu clave>.")
    print("     Copia .env.example si aun no lo tienes.")
    print("  2. Si acabas de crearlo o de editarlo: reinicia el kernel.")
    print("  3. Como alternativa, en una celda:")
    print('       os.environ["GEMINI_API_KEY"] = "..."')
    print("\nEsta seccion se saltara. El resto del notebook no depende de ella.")

.env: encontrado en /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio/.env
claves cargadas: ninguna

Gemini disponible — modelo gemini-flash-lite-latest


In [7]:
# La instruccion de sistema, que es donde viven las restricciones de seguridad.
print(INSTRUCCION_SISTEMA)

Eres el asistente de redacción de PhysioVision, una app de apoyo a ejercicios de
rehabilitación de hombro. Tu único trabajo es reescribir una recomendación ya
validada para que suene cercana y concreta.

REGLAS INVIOLABLES:
1. No inventes consejo clínico. Reformula SOLO lo que te den en RECOMENDACION.
2. No diagnostiques, no menciones patologías, no sugieras ejercicios nuevos, no
   propongas cambiar series, repeticiones ni cargas.
3. Puedes citar las métricas que te den para hacer el mensaje concreto.
4. Dirígete al paciente de tú, en español, con tono cálido y sereno.
5. Nunca alarmes. Si el resultado es un error de técnica, enmárcalo como un
   ajuste, no como un fallo.
6. Sin markdown, sin listas, sin emojis, sin comillas. Texto corrido.
7. Si los datos son contradictorios o insuficientes, limítate a reformular la
   recomendación sin citar números.



### 5.1 Comparacion sobre repeticiones reales

Se toma una repeticion de cada clase, de las que el modelo acaba de clasificar, y
se enfrenta el texto del JSON con el redactado. Es la forma de ver si el prompt
produce algo util o solo adorno.

In [8]:
# Requiere el dataset de repeticiones (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_DATASET:
    print("Se salta: falta el dataset de repeticiones.")
else:
    # `kb` y `EJERCICIO_KB` se cargaron en la seccion 2.

    # Una repeticion representativa por clase, de las predicciones out-of-fold.
    ejemplos = []
    for clase in sorted(df_reps.etiqueta.unique()):
        sub = df_reps[df_reps.etiqueta == clase]
        if not len(sub):
            continue
        fila = sub.iloc[len(sub) // 2]
        ejemplos.append((clase, fila))

    # Tras cada repeticion se muestra una CONSIGNA corta, no un parrafo: quien acaba
    # de moverse tiene un par de segundos antes de la siguiente. El texto largo se
    # reserva para el resumen del final de la serie.
    for clase, fila in ejemplos:
        consigna = kb.consigna(EJERCICIO_KB, clase)
        print("=" * 74)
        print(f"CLASE: {clase}   ({fila.sujeto}, repeticion {int(fila.idx_rep)})")
        print(f"  rom_max {fila.rom_max:.0f}  tronco_max {fila.tronco_max:.0f}  "
              f"duracion {fila.duracion_s:.1f}s")
        print("-" * 74)
        print(f"CONSIGNA BASE (json)   : {consigna}")
        if redactor.disponible:
            texto = redactor.redactar_repeticion(
                indice=int(fila.idx_rep) + 1, etiqueta=clase, confianza=0.9,
                recomendacion=consigna, variables=fila.to_dict(), lado=fila.lado)
            palabras = len(texto.split()) if texto else 0
            print(f"CONSIGNA REDACTADA (ia): {texto or '(sin respuesta)'}"
                  + (f"   [{palabras} palabras]" if texto else ""))
    print("=" * 74)
    print("\nSi alguna redaccion pasa de 6 palabras o deja de ser imperativa,")
    print("endurece PROMPT_REPETICION en src/gemini_feedback.py.")

CLASE: compensacion_tronco   (PM_109, repeticion 1)
  rom_max 108  tronco_max 20  duracion 8.4s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : Mantén el torso recto
CONSIGNA REDACTADA (ia): Mantén el torso recto   [4 palabras]
CLASE: correcto   (PM_032, repeticion 2)
  rom_max 18  tronco_max 7  duracion 2.9s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : ¡Correcto!
CONSIGNA REDACTADA (ia): ¡Bien hecho!   [2 palabras]
CLASE: rango_insuficiente   (PM_114, repeticion 19)
  rom_max 71  tronco_max 12  duracion 5.5s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : Sube más el brazo
CONSIGNA REDACTADA (ia): Eleva un poco más el brazo   [6 palabras]

Si alguna redaccion pasa de 6 palabras o deja de ser imperativa,
endurece PROMPT_REPETICION en src/gemini_feedback.py.


### 5.2 Resumen de serie

El mensaje de cierre tiene mas contexto que el de una repeticion: la secuencia
completa. Con ella el modelo puede notar cosas que ninguna repeticion aislada
muestra, como que los errores se concentren al final por fatiga.

In [9]:
# Requiere el dataset de repeticiones (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_DATASET:
    print("Se salta: falta el dataset de repeticiones.")
else:
    # Serie sintetica a partir de las repeticiones reales de un sujeto.
    sujeto_demo = df_reps.sujeto.iloc[0]
    serie = df_reps[(df_reps.sujeto == sujeto_demo) & (df_reps.vista == "frontal")]

    if len(serie):
        etiquetas = serie.etiqueta.tolist()
        errores = [e for e in etiquetas if e != "correcto"]
        dominante = (max(set(errores), key=errores.count)
                     if len(errores) >= max(1, len(etiquetas) / 3) else "correcto")
        resumen_demo = {
            "repeticiones": len(etiquetas),
            "correctas": sum(1 for e in etiquetas if e == "correcto"),
            "clasificacion": dominante,
            "por_repeticion": etiquetas,
            "rom_max": float(serie.rom_max.max()),
            "rom_medio": float(serie.rom_max.mean()),
            "lado": serie.lado.iloc[0],
        }
        conocimiento = kb.retrieve(EJERCICIO_KB, dominante)
        print(f"Sujeto {sujeto_demo}: {resumen_demo['repeticiones']} repeticiones")
        print(f"  secuencia: {' -> '.join(etiquetas)}")
        print(f"  dominante: {dominante}\n")
        print("TEXTO BASE (json):")
        print(f"  {conocimiento['recomendacion']}\n")
        if redactor.disponible:
            print("TEXTO REDACTADO (gemini):")
            print(f"  {redactor.redactar_resumen(resumen=resumen_demo, recomendacion=conocimiento['recomendacion']) or '(sin respuesta)'}")

Sujeto PM_000: 4 repeticiones
  secuencia: compensacion_tronco -> correcto -> correcto -> compensacion_tronco
  dominante: compensacion_tronco

TEXTO BASE (json):
  Mantén el torso vertical, activa suavemente el abdomen y reduce el rango si necesitas inclinarte para elevar el brazo.

TEXTO REDACTADO (gemini):
  Has completado esta serie con cuatro repeticiones con tu brazo derecho, logrando mantener un buen movimiento en el centro. A medida que avanzaba el ejercicio, se ha notado algo de cansancio y el torso ha tendido a inclinarse al final. Para la próxima vez, intenta activar suavemente el abdomen y mantén la espalda bien vertical, reduciendo un poco el rango si lo necesitas para asegurar que el movimiento sea cómodo y controlado.


### 5.3 Revision de seguridad de los textos generados

Antes de poner esto delante de un paciente conviene comprobar que el modelo se
cine a las reglas. Se generan varias redacciones de la misma repeticion y se
buscan senales de que se ha salido del guion: terminos diagnosticos, indicaciones
de dosis o promesas de recuperacion.

Es una comprobacion de humo, no una garantia. Si vas a usarlo con pacientes
reales, revisa una muestra a mano.

In [10]:
# Requiere el dataset de repeticiones (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_DATASET:
    print("Se salta: falta el dataset de repeticiones.")
else:
    # La consigna debe ser corta e imperativa; ademas no puede salirse del guion.
    TERMINOS_PROHIBIDOS = [
        # diagnostico
        "tendinitis", "bursitis", "desgarro", "lesion", "artrosis", "sindrome",
        "patologia", "diagnostic", "inflamacion",
        # dosis y prescripcion
        "series", "kilos", "kg", "peso", "mancuerna", "banda elastica",
        "veces al dia", "tres veces", "descansa 48",
        # promesas
        "te curaras", "en dos semanas", "garantiza", "desaparecera",
    ]

    if redactor.disponible and ejemplos:
        clase, fila = ejemplos[-1]
        conocimiento = kb.retrieve(EJERCICIO_KB, clase)
        print(f"Generando 5 redacciones de la misma repeticion ({clase})...\n")
        hallazgos = 0
        for i in range(5):
            texto = redactor.redactar_repeticion(
                indice=1, etiqueta=clase, confianza=0.88,
                recomendacion=conocimiento["recomendacion"],
                variables=fila.to_dict(), lado=fila.lado)
            if not texto:
                continue
            encontrados = [t for t in TERMINOS_PROHIBIDOS if t in texto.lower()]
            marca = "!!" if encontrados else "OK"
            hallazgos += len(encontrados)
            print(f"{marca} {texto}")
            if encontrados:
                print(f"     terminos fuera de guion: {encontrados}")
            print(f"     ({len(texto.split())} palabras)")
        print(f"\nTotal de terminos fuera de guion: {hallazgos}")
        if hallazgos:
            print("Endurece INSTRUCCION_SISTEMA en src/gemini_feedback.py y repite.")
    else:
        print("Sin Gemini disponible: nada que revisar.")

Generando 5 redacciones de la misma repeticion (rango_insuficiente)...

OK Eleva un poco más el brazo.
     (6 palabras)
OK Sube más el brazo
     (4 palabras)
OK Sube más el brazo.
     (4 palabras)
OK Sube más el brazo
     (4 palabras)
OK Sube más el brazo
     (4 palabras)

Total de terminos fuera de guion: 0


### 5.4 Coste y latencia

Una llamada por repeticion tiene coste. Con series de 15-20 repeticiones y varios
pacientes al dia conviene tener el numero delante antes de desplegar.

En la app la llamada de repeticion es **asincrona**: el mensaje del JSON aparece
al instante y el redactado lo sustituye cuando llega. Si tarda mas que el timeout
o falla, el paciente ve el texto base y no nota nada roto.

In [11]:
# Requiere el dataset de repeticiones (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_DATASET:
    print("Se salta: falta el dataset de repeticiones.")
else:
    import time

    if redactor.disponible and ejemplos:
        clase, fila = ejemplos[0]
        conocimiento = kb.retrieve(EJERCICIO_KB, clase)
        tiempos = []
        for _ in range(3):
            t0 = time.time()
            redactor.redactar_repeticion(
                indice=1, etiqueta=clase, confianza=0.9,
                recomendacion=conocimiento["recomendacion"],
                variables=fila.to_dict(), lado=fila.lado)
            tiempos.append(time.time() - t0)
        print(f"Latencia por repeticion: {np.mean(tiempos):.2f}s "
              f"(min {min(tiempos):.2f}, max {max(tiempos):.2f})")
        print(f"Una serie de 15 repeticiones = 15 llamadas + 1 de resumen")
        reps_totales = len(df_reps)
        print(f"Reprocesar las {reps_totales} repeticiones de este dataset serian "
              f"{reps_totales} llamadas (~{reps_totales * np.mean(tiempos) / 60:.0f} min)")
    else:
        print("Sin Gemini disponible: no se puede medir la latencia.")

Latencia por repeticion: 0.81s (min 0.40, max 1.61)
Una serie de 15 repeticiones = 15 llamadas + 1 de resumen
Reprocesar las 334 repeticiones de este dataset serian 334 llamadas (~4 min)


---
## 6. Voz de las consignas con Google Cloud Text-to-Speech

Quien eleva el brazo tiene la vista en su propio hombro, no en la pantalla. El
rotulo sobre el video ayuda, pero una consigna **dicha en voz alta** es lo que de
verdad se parece a tener un fisioterapeuta al lado.

### El diseno esta centrado en la cache, y por una razon concreta

Las consignas son un **conjunto cerrado y pequeno**: las de
`knowledge_base/ejercicios.json` son diez en total, y se repiten en cada serie de
cada paciente. Sintetizarlas una vez y guardarlas en disco convierte la
reproduccion en una lectura de archivo: latencia nula, coste nulo y funciona sin
red. Solo las consignas que redacta Gemini, que varian, pueden provocar una
sintesis nueva — y tambien se cachean.

Esta celda precalienta esa cache. Ejecutarla una vez deja la app lista.

### Autenticacion: no vale la clave de Gemini

Verificado contra el servicio: Cloud TTS responde
`401 UNAUTHENTICATED — API keys are not supported by this API`. Necesita
credenciales de **cuenta de servicio**, que es un alta distinta:

1. En la consola de Google Cloud, habilita *Cloud Text-to-Speech API*.
2. Crea una cuenta de servicio y descarga su JSON.
3. Anade al `.env` la ruta de ese archivo:

```
GOOGLE_APPLICATION_CREDENTIALS=/ruta/a/credenciales.json
```

Sin esto todo funciona igual, pero en silencio.

In [12]:
from src.voz import SintetizadorVoz, consignas_del_ejercicio

voz = SintetizadorVoz()
print(f"voz    : {voz.idioma}"
      + (f" ({voz.voz})" if voz.voz else " (voz por defecto)")
      + f", velocidad {voz.velocidad}")
print(f"cache  : {voz.directorio}")

# Hay dos motores posibles (Cloud TTS y la API de Gemini) y conviene saber cual
# quedo activo: la cuota y la calidad no son las mismas.
if voz.disponible:
    print(f"\nsintesis disponible — motor activo: {voz.motor_activo}")
else:
    print(f"\nsintesis NO disponible:\n  {voz.motivo_no_disponible}")
    print("  Solo se oira lo que ya este en cache.")

consignas = consignas_del_ejercicio(EJERCICIO_KB)
print(f"\n{len(consignas)} consignas en la base de conocimiento:")
for c in consignas:
    marca = "cacheada" if voz.en_cache(c) else "pendiente"
    print(f"  [{marca:9s}] {c}")

voz    : es-US (voz por defecto), velocidad 1.05
cache  : /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio/data/audio

sintesis disponible — motor activo: gemini

10 consignas en la base de conocimiento:
  [cacheada ] Sube más el brazo
  [cacheada ] Eleva un poco más
  [cacheada ] Busca más recorrido
  [cacheada ] Mantén el torso recto
  [cacheada ] No inclines el cuerpo
  [cacheada ] Activa el abdomen
  [cacheada ] ¡Correcto!
  [pendiente] ¡Bien hecho!
  [cacheada ] ¡Así es!
  [pendiente] ¡Muy bien!


In [13]:
# Precalentado. Idempotente: lo ya sintetizado no se vuelve a pedir.
if voz.disponible or any(voz.en_cache(c) for c in consignas):
    rutas = voz.precalentar(consignas)
    fallidas = [t for t, r in rutas.items() if r is None]
    print(f"{len(rutas) - len(fallidas)}/{len(rutas)} consignas con audio")
    if fallidas:
        print(f"  sin audio: {fallidas}")
    print(f"\nestadisticas: {voz.estadisticas()}")
else:
    print("Sin credenciales y sin cache previa: no hay nada que precalentar.")

La síntesis de voz falló ('NoneType' object has no attribute 'parts'); la sesión sigue en silencio


10/10 consignas con audio

estadisticas: {'sintesis': 2, 'aciertos_cache': 8, 'archivos_en_cache': 11}


### 6.1 Escuchar el resultado

Si hay audio, esta celda lo reproduce dentro del notebook. Merece la pena oirlo
antes de ponerlo delante de un paciente: la velocidad y la voz cambian bastante
la sensacion, y ambas se ajustan por variable de entorno
(`PHYSIOVISION_TTS_VELOCIDAD`, `PHYSIOVISION_TTS_VOZ`).

In [14]:
from IPython.display import Audio, display

reproducidas = 0
for consigna in consignas:
    ruta = voz.ruta_cache(consigna)
    if not ruta.exists():
        continue
    print(f"{consigna}   ({ruta.stat().st_size / 1024:.0f} KB)")
    display(Audio(str(ruta)))
    reproducidas += 1

if not reproducidas:
    print("No hay audio en la cache todavia. Ejecuta la celda anterior con")
    print("GOOGLE_APPLICATION_CREDENTIALS configurado.")

Sube más el brazo   (94 KB)


Eleva un poco más   (87 KB)


Busca más recorrido   (96 KB)


Mantén el torso recto   (104 KB)


No inclines el cuerpo   (100 KB)


Activa el abdomen   (91 KB)


¡Correcto!   (47 KB)


¡Bien hecho!   (59 KB)


¡Así es!   (51 KB)


¡Muy bien!   (57 KB)


### 6.2 Coste

El precalentado son diez sintesis, una sola vez. A partir de ahi una sesion de
quince repeticiones no hace **ninguna** llamada a Cloud TTS, porque todas las
consignas salen de la cache.

La excepcion son las consignas redactadas por Gemini, que varian y provocan una
sintesis la primera vez que aparece cada texto. Aun asi el vocabulario converge
rapido: son variaciones sobre las mismas tres ideas.

In [15]:
n_cacheadas = sum(1 for c in consignas if voz.en_cache(c))
print(f"consignas fijas cacheadas : {n_cacheadas}/{len(consignas)}")
print(f"archivos en la cache      : {voz.estadisticas()['archivos_en_cache']}")
print(f"llamadas por serie de 15  : {0 if n_cacheadas == len(consignas) else 'depende'}"
      " (las consignas fijas ya no se sintetizan)")

peso = sum(f.stat().st_size for f in voz.directorio.glob("*.mp3"))
print(f"peso total de la cache    : {peso / 1024:.0f} KB")

consignas fijas cacheadas : 10/10
archivos en la cache      : 11
llamadas por serie de 15  : 0 (las consignas fijas ya no se sintetizan)
peso total de la cache    : 0 KB


---
## 7. PhysioVision en Gradio, en su forma mas simple

Todo lo anterior junto, con una interfaz: **subes un video, la app responde**. Es el
mismo pipeline de la seccion 4 mas la consigna de la seccion 5 y el audio de la 6.

Es deliberadamente el camino corto:

| Aqui | La app completa (`python app.py`) |
|---|---|
| video subido, se procesa entero al final | camara en vivo, frame a frame a 30 Hz |
| segmentacion sobre la senal completa | `src/segmentador_online.py`, repeticion a repeticion |
| sin estado ni historial | sesion en SQLite (`src/storage.py`) |
| una funcion en una celda | `ui/app_ui.py` + `ui/callbacks.py` |

Lo que **no** cambia entre las dos es lo que importa para el modelo: los mismos
artefactos de `models/`, el mismo contrato de variables y las mismas funciones de
`src/features_ex1.py`. Esta version sirve para ver el modelo funcionando de punta a
punta; la de `app.py` es la que se pone delante de un paciente.

Las dos casillas degradan solas: sin clave de Gemini se muestra el texto del JSON, sin
credenciales de TTS no hay audio, y en ninguno de los dos casos falla el analisis.

### 7.1 El callback: video -> clase -> consigna -> voz

In [16]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    def analizar_video(ruta_video: str | None, con_ia: bool, con_voz: bool):
        """Callback de la interfaz: video -> (resumen, tabla por repeticion, audio)."""
        if not ruta_video:
            return "Sube un video del ejercicio para analizarlo.", None, None

        try:
            salida = predecir_video(ruta_video)
        except ValueError as error:
            # Lo normal: video demasiado corto, o donde no se ve a la persona.
            return f"**No se pudo analizar.** {error}", None, None

        usa_ia = con_ia and redactor.disponible

        filas = []
        for r in salida["detalle"]:
            consigna = kb.consigna(EJERCICIO_KB, r["label"], indice=r["idx"])
            if usa_ia:
                # Si Gemini no responde a tiempo se queda la consigna del JSON.
                consigna = redactor.redactar_repeticion(
                    indice=r["idx"] + 1, etiqueta=r["label"], confianza=r["confidence"],
                    recomendacion=consigna, variables=r, lado=salida["lado"]) or consigna
            filas.append({
                "#": r["idx"] + 1,
                "clase": r["label"],
                "confianza": round(r["confidence"], 2),
                "rango (grados)": round(r["rom_max"]),
                "tronco (grados)": round(r["tronco_max"]),
                "duracion (s)": round(r["duracion_s"], 1),
                "consigna": consigna,
            })

        conocimiento = kb.retrieve(EJERCICIO_KB, salida["label"])
        cierre = conocimiento["recomendacion"]
        if usa_ia:
            cierre = redactor.redactar_resumen(
                resumen={"repeticiones": salida["n_repeticiones"],
                         "correctas": salida["correctas"],
                         "clasificacion": salida["label"],
                         "por_repeticion": salida["por_repeticion"],
                         "rom_max": salida["rom_max"],
                         "rom_medio": salida["rom_medio"],
                         "lado": salida["lado"]},
                recomendacion=cierre) or cierre

        audio = None
        if con_voz and (voz.disponible or voz.en_cache(cierre)):
            ruta = voz.sintetizar(cierre)
            audio = str(ruta) if ruta else None

        lado_legible = {"left": "izquierdo", "right": "derecho"}.get(salida["lado"], "?")
        resumen = f"""### {conocimiento["titulo"]}

    **{salida["correctas"]}/{salida["n_repeticiones"]} repeticiones correctas** ·
    lado {lado_legible} · rango maximo {salida["rom_max"]:.0f} grados

    {cierre}

    _{conocimiento["precaucion"]}_

    <sub>clase de la serie: `{salida["label"]}` ({salida["confidence"]:.0%} de confianza) ·
    modelo: `{salida["source"]}` · texto: {"Gemini" if usa_ia else "knowledge_base"}</sub>
    """
        return resumen, pd.DataFrame(filas), audio

### 7.2 La interfaz

In [17]:
import gradio as gr

ESTADO = (f"modelo `{RUTA_MODELO.name}` (macro-F1 "
          f"{contrato['metricas_loso']['macro_f1']:.2f}) · "
          f"Gemini {'disponible' if redactor.disponible else 'no disponible'} · "
          f"voz {'disponible' if voz.disponible else 'solo cache'}")

with gr.Blocks(title="PhysioVision — Ex1") as demo:
    gr.Markdown(f"""
    # PhysioVision — elevacion lateral de hombro
    Sube un video del ejercicio. Se segmenta en repeticiones y cada una se clasifica
    con el modelo entrenado en el primer notebook.

    <sub>{ESTADO}</sub>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            entrada = gr.Video(label="Video del ejercicio", sources=["upload"])
            con_ia = gr.Checkbox(
                value=redactor.disponible, interactive=redactor.disponible,
                label="Redactar la realimentacion con Gemini",
                info="Sin clave se usa el texto de knowledge_base/ejercicios.json.")
            con_voz = gr.Checkbox(
                value=voz.disponible, interactive=True,
                label="Decir la recomendacion en voz alta",
                info="Sin credenciales de TTS solo suena lo que ya este en cache.")
            boton = gr.Button("Analizar", variant="primary")
        with gr.Column(scale=1):
            salida_resumen = gr.Markdown()
            salida_audio = gr.Audio(label="Recomendacion", type="filepath",
                                    autoplay=True)
            salida_tabla = gr.Dataframe(label="Repeticiones", wrap=True)

    # El orden de esta lista es el de lo que devuelve analizar_video()
    # (resumen, tabla, audio), no el del layout de arriba.
    boton.click(analizar_video, [entrada, con_ia, con_voz],
                [salida_resumen, salida_tabla, salida_audio])

print("Interfaz construida. La celda siguiente la levanta.")

Interfaz construida. La celda siguiente la levanta.


### 7.3 Levantar la app

In [ ]:
# Se levanta dentro del notebook. En Colab no hay navegador local al que
# asomarse, asi que se pide un enlace publico temporal; en local, no.
# Para pararla, ejecuta la celda siguiente.
demo.queue(default_concurrency_limit=2).launch(
    inline=not EN_COLAB, share=EN_COLAB, quiet=True, height=900)


: 

### 7.4 Parar el servidor

In [19]:
demo.close()
print("Servidor detenido. La app completa se levanta con:  python app.py")

Closing server running on port: 7860
Servidor detenido. La app completa se levanta con:  python app.py


---
## 8. De aqui a la app real

Lo que acaba de correr consume exactamente los artefactos del primer notebook. Para
llevarlo a la aplicacion que se usa de verdad:

```bash
python app.py            # camara en vivo, historial y sesiones
```

`ui/callbacks.py` hace lo mismo que `analizar_video()` pero por frame, y `src/`
sostiene las dos rutas: mismo `ExerciseClassifier`, mismo contrato, mismas variables.

### Que queda pendiente

1. **Vista lateral.** Aqui se asume frontal (`es_lateral = 0`). El modelo tiene la
   variable y el dataset tiene las dos vistas: falta decidir en la interfaz cual es.
2. **Cambio de dominio.** El modelo se entreno con camaras fijas de laboratorio y la
   app recibe video de movil. El rendimiento real sera menor que el macro-F1 del
   contrato.
3. **Latencia de Gemini.** En esta version las llamadas son sincronas y una serie de
   quince repeticiones se nota. La app las hace en segundo plano y muestra primero el
   texto del JSON.
4. **Etiquetas.** Si el contrato dice `etiquetado: reglas_automaticas`, las metricas
   miden consistencia interna, no validez clinica. Revisa
   `data/datasets/ex1_etiquetas.csv` y reentrena.